## Load CHIA + helper

In [3]:
from datasets import load_dataset
from pathlib import Path
from collections import defaultdict
import json
import random

# Find repository root from the current notebook location
ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "README.md").exists() and (p / "scripts").exists():
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError("Could not find clinical-trial-eligibility-rule-verification project root.")

OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Output directory:", OUT_DIR)

Project root: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification
Output directory: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed


In [7]:
N_DOCS = 100   # number of CHIA trial documents to sample
SEED = 11      # random seed for reproducibility

# -----------------------------
# Load CHIA
# -----------------------------
from datasets import load_dataset

ds = load_dataset("bigbio/chia", "chia_bigbio_kb")

print(ds)
print(ds.keys())

Using the latest cached version of the dataset since bigbio/chia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'chia_bigbio_kb' at C:\Users\JUlloa\.cache\huggingface\datasets\bigbio___chia\chia_bigbio_kb\1.0.0\36e5df0d60dfc5152cd22a807ade73f135105008 (last modified on Mon Feb 23 14:00:31 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 2000
    })
})
dict_keys(['train'])


In [8]:
ds = load_dataset("bigbio/chia", "chia_bigbio_kb", split="train")

print(ds)
print(len(ds))

Using the latest cached version of the dataset since bigbio/chia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'chia_bigbio_kb' at C:\Users\JUlloa\.cache\huggingface\datasets\bigbio___chia\chia_bigbio_kb\1.0.0\36e5df0d60dfc5152cd22a807ade73f135105008 (last modified on Mon Feb 23 14:00:31 2026).


Dataset({
    features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
    num_rows: 2000
})
2000


## Function to build a subset with a filter

In [10]:
def get_text(ex):
    passage = ex["passages"][0]
    txt = passage["text"]
    if isinstance(txt, list):
        txt = " ".join(txt)
    return (txt or "").strip()

# -----------------------------
# Build valid rows
# -----------------------------
valid_rows = []

for ex in ds:
    txt = get_text(ex)

    if not txt:
        continue

    if len(ex["entities"]) == 0:
        continue

    if len(ex["relations"]) == 0:
        continue

    valid_rows.append({
        "chia_id": ex["id"],
        "document_id": ex["document_id"],
        "text": txt,
        "n_entities": len(ex["entities"]),
        "n_relations": len(ex["relations"]),
    })

print("Valid rows:", len(valid_rows))

Valid rows: 1876


In [11]:
# -----------------------------
# Group by document_id
# -----------------------------
rows_by_doc = defaultdict(list)

for row in valid_rows:
    rows_by_doc[row["document_id"]].append(row)

# Keep only documents that have both inclusion and exclusion rows
eligible_docs = []

for doc_id, rows in rows_by_doc.items():
    chia_ids = {r["chia_id"] for r in rows}

    has_inc = any(cid.endswith("_inc") for cid in chia_ids)
    has_exc = any(cid.endswith("_exc") for cid in chia_ids)

    if has_inc and has_exc:
        eligible_docs.append(doc_id)

print("Eligible documents with inclusion and exclusion:", len(eligible_docs))

Eligible documents with inclusion and exclusion: 886


In [12]:
# -----------------------------
# Random fixed sample of documents
# -----------------------------
random.seed(SEED)
random.shuffle(eligible_docs)

selected_doc_ids = eligible_docs[:N_DOCS]
selected_doc_ids_set = set(selected_doc_ids)

selected_rows = []

for doc_id in selected_doc_ids:
    rows = rows_by_doc[doc_id]

    # sort so inclusion usually appears before exclusion
    rows = sorted(rows, key=lambda r: r["chia_id"])

    for row in rows:
        selected_rows.append(row)

print("Selected documents:", len(selected_doc_ids))
print("Selected rows:", len(selected_rows))


Selected documents: 100
Selected rows: 200


In [13]:
# -----------------------------
# Save text-only extraction file
# -----------------------------
text_only_path = OUT_DIR / "chia_text_only_200.jsonl"

with open(text_only_path, "w", encoding="utf-8") as f:
    for row in selected_rows:
        out = {
            "chia_id": row["chia_id"],
            "document_id": row["document_id"],
            "text": row["text"],
        }
        f.write(json.dumps(out) + "\n")

print("Saved:", text_only_path)

Saved: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_text_only_200.jsonl


In [14]:
# -----------------------------
# Save structural evaluation file
# Same selected rows, with counts
# -----------------------------
struct_path = OUT_DIR / "chia_struct_eval_200.jsonl"

with open(struct_path, "w", encoding="utf-8") as f:
    for row in selected_rows:
        out = {
            "chia_id": row["chia_id"],
            "document_id": row["document_id"],
            "text": row["text"],
            "n_entities": row["n_entities"],
            "n_relations": row["n_relations"],
        }
        f.write(json.dumps(out) + "\n")

print("Saved:", struct_path)

Saved: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_struct_eval_200.jsonl


In [15]:
# -----------------------------
# Save selected IDs for reproducibility
# -----------------------------
ids_path = OUT_DIR / "chia_selected_ids_200.json"

with open(ids_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "source_dataset": "bigbio/chia",
            "source_config": "chia_bigbio_kb",
            "source_split": "train",
            "source_rows": len(ds),
            "sampled_by": "document_id",
            "selected_documents": len(selected_doc_ids),
            "selected_rows": len(selected_rows),
            "random_seed": SEED,
            "selected_document_ids": selected_doc_ids,
            "selected_chia_ids": [r["chia_id"] for r in selected_rows],
        },
        f,
        indent=2,
    )

print("Saved:", ids_path)

Saved: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed\chia_selected_ids_200.json


In [17]:
# -----------------------------
# Sanity checks
# -----------------------------
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

text_rows = read_jsonl(text_only_path)
struct_rows = read_jsonl(struct_path)

text_ids = [r["chia_id"] for r in text_rows]
struct_ids = [r["chia_id"] for r in struct_rows]

print("Text-only rows:", len(text_rows))
print("Structural rows:", len(struct_rows))
print("Same IDs:", text_ids == struct_ids)
print("Unique documents:", len(set(r["document_id"] for r in text_rows)))
print("Output folder:", OUT_DIR)

Text-only rows: 200
Structural rows: 200
Same IDs: True
Unique documents: 100
Output folder: c:\Users\JUlloa\Documents\MSc Statistics and Data Science\Master Thesis\clinical-trial-eligibility-rule-verification\data\processed
